In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
from langchain_core.tools import tool


@tool
def publish_article(slug: str) -> str:
    """Publish an article to the web."""
    return f"✅ Published {slug}"


@tool
def list_articles() -> str:
    """List all articles (read-only)."""
    return "article-1\narticle-2\narticle-3"


print("Tools defined")

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="claude-haiku-4-5",
    tools=[publish_article, list_articles],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"publish_article": {"allowed_decisions": ["approve", "reject"]}},
        ),
    ],
)

print("Agent with HITL middleware created")

In [ ]:
# Test 1: Read-only (no interrupt)
print("\n" + "=" * 60)
print("TEST 1: List articles (no approval needed)")
print("=" * 60)

config = {"configurable": {"thread_id": "1"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "List my articles"}]},
    config=config,
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")

In [ ]:
# Test 2: Sensitive operation (interrupts)
print("\n" + "=" * 60)
print("TEST 2: Publish article (requires approval)")
print("=" * 60)

config = {"configurable": {"thread_id": "2"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Publish article-1"}]},
    config=config,
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")

if "__interrupt__" in result:
    desc = result["__interrupt__"][-1].value["action_requests"][-1]["description"]
    print(f"\n🔒 INTERRUPT: {desc}")

In [ ]:
from langgraph.types import Command

# Test 3: Approve and resume
print("\n" + "=" * 60)
print("TEST 3: Approve and resume")
print("=" * 60)

result = agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config,  # Same thread
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")

In [ ]:
# Test 4: Reject request
print("\n" + "=" * 60)
print("TEST 4: Reject publication")
print("=" * 60)

config = {"configurable": {"thread_id": "3"}}
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Publish article-2"}]},
    config=config,
)

for msg in result["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")

if "__interrupt__" in result:
    print("\n🔒 Approval required")
    result = agent.invoke(
        Command(resume={"decisions": [{"type": "reject", "message": "Not ready yet"}]}),
        config=config,
    )

    for msg in result["messages"]:
        print(f"\n{msg.__class__.__name__}:")
        print(f"{msg.content}")